# Final submission

Trains the CV-selected best model (`notebooks/modeling.ipynb`: LightGBM, `+ rolling stats` feature set, 0.5212 TS-grouped CV accuracy) on the full training set and writes a submission file.

## Setup

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import lightgbm as lgbm

import qrt_prep as P
import qrt_features as F

## Load, engineer, train

In [2]:
X_train, y_train, X_test = P.load_raw('../data/raw')
train_df = F.engineer(X_train.join(y_train))
test_df = F.engineer(X_test)

FEATURES = P.BASE_FEATURES + F.GROUP_DUMMY_COLS + F.MISSING_COLS + F.ROLLING_COLS
len(FEATURES)

49

In [3]:
params = {'objective': 'mse', 'metric': 'mse', 'verbosity': -1, 'seed': 42,
          'learning_rate': 1e-2, 'max_depth': 3}
train_set = lgbm.Dataset(F.to_matrix(train_df, FEATURES), label=train_df['target'].to_numpy())
model = lgbm.train(params, train_set, num_boost_round=300)

## Predict and format

In [4]:
preds = model.predict(F.to_matrix(test_df, FEATURES))
submission = pd.DataFrame({'prediction': (preds > 0).astype(int)}, index=test_df.index)
submission.index.name = 'ROW_ID'
submission['prediction'].value_counts(normalize=True)

prediction
1    0.57176
0    0.42824
Name: proportion, dtype: float64

## Sanity checks against sample_submission.csv

In [5]:
sample_submission = pd.read_csv('../data/raw/sample_submission.csv', index_col='ROW_ID')

assert submission.shape == sample_submission.shape
assert (submission.index == sample_submission.index).all()
assert set(submission['prediction'].unique()) <= {0, 1}
print('shape ok, index aligned, values in {0, 1}')
print('predicted positive share:', submission['prediction'].mean())
print('sample_submission positive share (random):', sample_submission['prediction'].mean())

shape ok, index aligned, values in {0, 1}
predicted positive share: 0.5717602761217446
sample_submission positive share (random): 0.5014747411358644


## Write

In [6]:
submission.to_csv('../submissions/lightgbm_rolling_stats.csv')
pd.read_csv('../submissions/lightgbm_rolling_stats.csv').head()

,ROW_ID,prediction
0,527073,0
1,527074,1
2,527075,0
3,527076,0
4,527077,0


## Threshold sanity check

Out-of-fold predictions on train are skewed positive (59% > 0, vs. the true 50.7% base rate) -- worth checking whether thresholding at exactly 0 is actually accuracy-optimal, or whether the model's regression output needs recentering before taking the sign. Tested with a double-CV: for each fold, pick the accuracy-maximizing threshold using only the *other* folds' OOF predictions, then score the held-out fold with it (so the threshold never sees the labels it's evaluated against).

In [7]:
folds = P.make_folds(train_df['TS'], n_splits=5, seed=0)
y = train_df['target'].to_numpy()
y_sign = (y > 0).astype(int)
oof = np.zeros(len(train_df))

for fold in range(5):
    val_mask = folds == fold
    fold_train = lgbm.Dataset(F.to_matrix(train_df[~val_mask], FEATURES), label=y[~val_mask])
    fold_model = lgbm.train(params, fold_train, num_boost_round=300)
    oof[val_mask] = fold_model.predict(F.to_matrix(train_df[val_mask], FEATURES))

grid = np.quantile(oof, np.linspace(0.01, 0.99, 200))
acc_t0, acc_tuned = [], []
for fold in range(5):
    val_mask = folds == fold
    other_pred, other_y = oof[~val_mask], y_sign[~val_mask]
    t_star = grid[np.argmax([((other_pred > t).astype(int) == other_y).mean() for t in grid])]
    acc_t0.append(((oof[val_mask] > 0).astype(int) == y_sign[val_mask]).mean())
    acc_tuned.append(((oof[val_mask] > t_star).astype(int) == y_sign[val_mask]).mean())

print('OOF predicted positive share:', (oof > 0).mean(), ' true positive share:', y_sign.mean())
print(f'threshold=0     mean acc: {np.mean(acc_t0):.4f}')
print(f'tuned threshold mean acc: {np.mean(acc_tuned):.4f}')

OOF predicted positive share: 0.5907530835387128  true positive share: 0.5071840143585423
threshold=0     mean acc: 0.5208
tuned threshold mean acc: 0.5207


Tuning makes no difference (0.5207 vs 0.5207, tuned is marginally *worse*): the skew doesn't hurt accuracy, because accuracy depends on where the sign genuinely flips relative to truth, not on matching the marginal predicted-positive rate to the true one. `sign(pred) > 0` is already the right rule -- the submission above needs no threshold correction.